In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd


import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns


/usr/workspace/pandey2/DFT/envDFT/lib/python3.9/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [ ]:
use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph_hashed import DFGrepInterference, DFGrepWorkflow 

if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None

# App Name
app_name = "cm1" #cosmoflow cm1


condition_fn = None #

if app_name == "cm1":
    # filename = "/p/lustre3/pandey2/logs/cm1/*.pfw.gz"
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/cm1/APP/node-32/v1/COMPACT/*.pfw.gz"
    cp_dir = "/p/lustre3/pandey2/logs/result_checkpoint/c"

elif app_name == "cosmoflow":
    # filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/cosmoflow/dlio-v100/node-16/v1/COMPACT/*.pfw.gz"
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.5-develop/corona/cosmoflow/dlio-v100/node-4/v1/COMPACT/*.pfw.gz"
    cp_dir =  "/p/lustre3/pandey2/logs/result_checkpoint"

elif app_name == "deepspeed-dlio-step100":
    filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-step100/node-16/v1/COMPACT/*.pfw.gz"
    cp_dir =  "/p/lustre3/pandey2/logs/result_checkpoint"
else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, debug=True,
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=True, 
                                host_pattern=r'lassen(\d+)', time_granularity=1e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()

[INFO] [07:02:46] Initialized Client with 64 workers and link http://134.9.71.20:8787/status [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:769]


/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


[INFO] [07:02:51] Restarting all workers [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:761]


In [4]:
# reset_dask_cluster()
# filename = "/p/lustre3/iopp/dftracer-traces-lfs/v1.0.6-develop/corona/megatron-deepspeed/dlio-step100/node-16/v1/COMPACT/*1*.pfw.gz"

In [5]:
def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))


In [6]:
def cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return '/'.join(path.split('/', 3)[:3])

    if "M" == json_object["ph"] and "FH" == json_object["name"] and "args" in json_object and "name" in json_object["args"]:
        d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=json_object["args"]["name"])
    if "args" in json_object and "M" != json_object["ph"]:
        if "ret" in json_object["args"]:
            d["size"] = int(json_object["args"]["ret"]) 
    return d

load_cols = {'size': "int64[pyarrow]" }
load_cols_metadata = {"FH":{'mount_point':"string[pyarrow]" }}

In [7]:
# def montage_cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
#     d = {}
#     def find_mount_point(path,trie):
#         mount_point = trie.longest_prefix(path)
#         if mount_point:
#             return mount_point.key
#         return '/'.join(path.split('/', 3)[:3])

#     if "M" == json_object["ph"] and "FH" == json_object["name"] and "args" in json_object and "name" in json_object["args"]:
#         d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=json_object["args"]["name"])
#     if "args" in json_object and "M" != json_object["ph"]:
#         if "ret" in json_object["args"]:
#             d["size"] = int(json_object["args"]["ret"]) 
#     return d

# # load_cols_montage = {'filename':"string[pyarrow]",'mount_point':"string[pyarrow]", 'size': "uint64[pyarrow]" }
# load_cols_montage = {'size': "int64[pyarrow]" }
# load_cols_montage_metadata = {"FH":{'mount_point':"string[pyarrow]" }}
# analyzer = DFAnalyzer(filename)
# analyzer.file_hash.head()
# analyzer.host_hash.head()
# analyzer.file_hash.head()


In [8]:
analyzer = DFAnalyzer(filename,load_fn=cols_function, load_cols=load_cols, load_data={"mount_point":trie}, metadata_cols = load_cols_metadata)

[INFO] [15:54:54] Created index for 1 files [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:428]
[INFO] [15:54:54] Total size of all files are <dask.bag.core.Item object at 0x1555416ccbe0> bytes [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:430]
[INFO] [15:54:54] test debug [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:433]
[INFO] [15:54:55] Loading 267 batches out of 1 files and has 4365520 lines overall [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:444]
[INFO] [15:55:10] Loaded events [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:511]
[INFO] [15:55:10] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:517]


In [9]:
df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]

df1['fhash'] = df1['fhash'].str.replace('.0', '', regex=False)  # Remove '.0'
df1['hhash'] = df1['hhash'].str.replace('.0', '', regex=False)  # Remove '.0'

df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
df3 = df3.rename(columns={'hash':'hhash'}) # update needed for this specific dataset

result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
analyze_df['id'] = analyze_df.index

In [18]:
analyze_df.compute().groupby("mount_point").size().reset_index(name="Count").sort_values(by="Count", ascending=False).head(10)

,mount_point,Count
6,/sys,2210114
5,/proc,456988
10,/usr/tce,203672
0,/dev,197851
1,/dev/shm,112932
3,/etc/libibverbs.d,43289
7,/tmp,42639
12,/var/tmp,42127
8,/usr/lib64,6282
26463,namelist.input,5132


In [10]:
filter_mp = ["/dev/shm", "/tmp","/var/tmp","/usr/tce", "/sys","/proc"] # significant missed /usr/workspace 2.8M usr WS2 237K

In [11]:
IFCalculator = DFGrepInterference(analyze_df, app_name=app_name, cp_dir=cp_dir, existing=False)

In [10]:
IFCalculator.ddf_data.groupby("mount_point").count().compute()

,id,name,cat,size,ts,te,dur,trange,hostname
mount_point,,,,,,,,,
/dev,56472,56472,56472,56472,56472,56472,56472,56472,56472
/dev/shm,69168,69168,69168,69168,69168,69168,69168,69168,69168
/proc,152629,152629,152629,152629,152629,152629,152629,152629,152629
/sys,372849,372849,372849,372849,372849,372849,372849,372849,372849
/tmp,1418,1418,1418,1418,1418,1418,1418,1418,1418
...,...,...,...,...,...,...,...,...,...
cm1out_000991_000003_w.dat,4,4,4,4,4,4,4,4,4
cm1out_000991_000005_s.dat,7,7,7,7,7,7,7,7,7
cm1out_000991_000006_v.dat,2,2,2,2,2,2,2,2,2


In [12]:
IFCalculator.ddf_data = IFCalculator.ddf_data[IFCalculator.ddf_data["mount_point"].isin(filter_mp)]
IFCalculator.ddf_metadata = IFCalculator.ddf_metadata[IFCalculator.ddf_metadata["mount_point"].isin(filter_mp)]

In [12]:
IFCalculator.ddf_data.groupby("mount_point").count().compute()

,id,name,cat,size,ts,te,dur,trange,hostname
mount_point,,,,,,,,,
/dev/shm,69168,69168,69168,69168,69168,69168,69168,69168,69168
/proc,152629,152629,152629,152629,152629,152629,152629,152629,152629
/sys,372849,372849,372849,372849,372849,372849,372849,372849,372849
/tmp,1418,1418,1418,1418,1418,1418,1418,1418,1418
/usr/tce,5273,5273,5273,5273,5273,5273,5273,5273,5273
/var/tmp,6941,6941,6941,6941,6941,6941,6941,6941,6941


In [13]:
IFCalculator.get_degree()

[INFO] [15:55:39]  Finding degree for data columns started ! [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/graph_hashed.py:114]


[INFO] [15:55:39]  Degree for data Completed ! [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/graph_hashed.py:118]
[INFO] [15:55:39]  Degree for metadata Started ! [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/graph_hashed.py:119]
[INFO] [15:55:39]  Degree for data Completed ! [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/graph_hashed.py:122]


In [14]:
IFCalculator.get_interference()

In [15]:
IFCalculator.write_checkpoint(id="inter", cp_dir = cp_dir)

In [16]:
IFCalculator.get_interference_metadata()

KilledWorker: Attempted to run task 'ndarray-46bd172f-4694-41bd-a6c1-4d2c2d6c0fca' on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://192.168.128.174:33923. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

In [ ]:
IFCalculator.write_checkpoint(id="inter_metadata", cp_dir = cp_dir)

In [18]:
IFCalculator.ddf_metadata.groupby(["mount_point","name"]).count().compute()

id    cat   size     ts     te    dur  trange  \
mount_point name                                                          
/dev/shm    __fxstat    1429   1429      0   1429   1429   1429    1429   
            __xstat     1103   1103      0   1103   1103   1103    1103   
            access      3072   3072      0   3072   3072   3072    3072   
            close      13940  13940  13940  13940  13940  13940   13940   
            ftruncate   8965   8965      0   8965   8965   8965    8965   
...                      ...    ...    ...    ...    ...    ...     ...   
/var/tmp    fopen64      320    320      0    320    320    320     320   
/proc       rmdir         19     19      0     19     19     19      19   
/usr/tce    fopen64       32     32      0     32     32     32      32   
            mkdir          2      2      0      2      2      2       2   
            rmdir          2      2      0      2      2      2       2   

                       hostname  
mount_point name                 
/dev/shm    __fxstat       1429  
            __xstat        1103  
            access         3072  
            close         13940  
            ftruncate      8965  
...                         ...  
/var/tmp    fopen64         320  
/proc       rmdir            19  
/usr/tce    fopen64          32  
            mkdir             2  
            rmdir             2  

[76 rows x 8 columns]

In [19]:
a=IFCalculator.inter.compute()

KeyboardInterrupt: 

In [15]:
IFCalculator.get_interference_metadata()

KeyboardInterrupt: 

In [25]:
data.groupby("mount_point").count()

,id,name,cat,size,ts,te,dur,trange,hostname,deg
mount_point,,,,,,,,,,
/dev/shm,69168,69168,69168,69168,69168,69168,69168,69168,69168,69168
/proc,152629,152629,152629,152629,152629,152629,152629,152629,152629,152629
/sys,372849,372849,372849,372849,372849,372849,372849,372849,372849,372849
/tmp,1418,1418,1418,1418,1418,1418,1418,1418,1418,1418
/usr/tce,5273,5273,5273,5273,5273,5273,5273,5273,5273,5273
/var/tmp,6941,6941,6941,6941,6941,6941,6941,6941,6941,6941


In [28]:
IFCalculator.ddf_metadata.groupby(["mount_point"]).count().compute()

KeyboardInterrupt: 

In [26]:
# data = IFCalculator.deg_data.compute()
metadata = IFCalculator.deg_metadata.compute()

KeyboardInterrupt: 

In [ ]:
IFCalculator.ddf_metadata.groupby("mount_point").count().compute()

In [ ]:
# data.deg.max()
# metadata.query('deg > 2000')
# IFCalculator.ddf_metadata.query('ts > 231869329 and te < 231887212').compute()

np.int64(692)

In [40]:
metadata.groupby(['trange','mount_point']).size().reset_index(name="Count").sort_values(by="Count",ascending = False).head(10)

,trange,mount_point,Count
207,52,/usr/WS2,163357
197,50,/usr/WS2,148461
202,51,/usr/WS2,122007
192,49,/usr/WS2,117745
247,67,/usr/WS2,104517
212,53,/usr/WS2,96700
509,469,/usr/WS2,71715
252,68,/usr/WS2,70528
19,4,/collab/usr/gapps,66996
23,4,/usr/WS2,64478


In [25]:
data.groupby("mount_point").count()

,id,name,cat,size,ts,te,dur,trange,hostname,deg
mount_point,,,,,,,,,,
/collab/usr/gapps,8928,8928,8928,8928,8928,8928,8928,8928,8928,8928
/dev,1152,1152,1152,1152,1152,1152,1152,1152,1152,1152
/dev/shm,109638,109638,109638,109638,109638,109638,109638,109638,109638,109638
/g/g92,72,72,72,72,72,72,72,72,72,72
/p/lustre3,26532,26532,26532,26532,26532,26532,26532,26532,26532,26532
/proc,4986,4986,4986,4986,4986,4986,4986,4986,4986,4986
/sys,52020,52020,52020,52020,52020,52020,52020,52020,52020,52020
/usr/WS2,237186,237186,237186,237186,237186,237186,237186,237186,237186,237186
/usr/share,54,54,54,54,54,54,54,54,54,54


In [ ]:
mount_points = ["/dev/shm", "/p/lustre3","/var/tmp", "/sys"] # significant missed /usr/workspace 2.8M usr WS2 237K
mount_points_metadata = ["/dev/shm", "/p/lustre3","/var/tmp", "/sys","/dev","/collab", "/usr/workspace"] #significant missed 2M usr WS2

In [27]:
metadata.groupby("mount_point").count()

,id,name,cat,size,ts,te,dur,trange,hostname,deg
mount_point,,,,,,,,,,
./,36,36,36,0,36,36,36,36,36,36
/collab,10422,10422,10422,0,10422,10422,10422,10422,10422,10422
/collab/usr,10422,10422,10422,0,10422,10422,10422,10422,10422,10422
/collab/usr/gapps,199854,199854,199854,17514,199854,199854,199854,199854,199854,199854
/dev,2862,2862,2862,558,2862,2862,2862,2862,2862,2862
/dev/shm,326779,326779,326779,54385,326779,326779,326779,326779,326779,326779
/etc/crypto-policies,144,144,144,0,144,144,144,144,144,144
/etc/libfabric.conf,36,36,36,0,36,36,36,36,36,36
/etc/libibverbs.d,756,756,756,0,756,756,756,756,756,756


In [ ]:
analyze_df.query()

In [26]:
# analyzer.file_hash.query('mount_point == "/usr/WS2"').compute()

analyzer.file_hash.query('mount_point == "/usr/workspace"').compute()


,name,pid,tid,hhash,mount_point
hash,,,,,
4.612157247353279e+18,/usr/workspace/haridev/iopp/software/scr-dlio,1042054,1042054,12578331172924968642,/usr/workspace
4.850557718170443e+18,/usr/workspace/haridev/iopp/apps/dlio,1042054,1042054,12578331172924968642,/usr/workspace
9.300293676376754e+18,/usr/workspace/iopp/projects/digio,1042054,1042054,12578331172924968642,/usr/workspace
1.4730023339222833e+19,/usr/workspace/haridev/iopp/apps/dlio/output/m...,1042054,1042054,12578331172924968642,/usr/workspace
3.546089249156215e+18,/usr/workspace/haridev/iopp/apps/dlio/output/m...,1042054,1042054,12578331172924968642,/usr/workspace
6.502414831719963e+18,/usr/workspace/haridev/iopp/apps/dlio/output,1042054,1042054,12578331172924968642,/usr/workspace
9.061657290076685e+18,/usr/workspace/haridev/iopp/apps/dlio/output/m...,1042054,1042054,12578331172924968642,/usr/workspace
1.2261448077921956e+19,/usr/workspace/haridev/iopp/apps/dlio/output/m...,1042054,1042054,12578331172924968642,/usr/workspace
1.0872469974753937e+19,/usr/workspace/haridev/iopp/apps/dlio/output/m...,1042055,1042055,12578331172924968642,/usr/workspace


In [ ]:
analyzer.file_hash.query('mount_point == "/usr/WS2"').

In [ ]:
t = analyzer.events.query('mount_point == "/usr/WS2"')

,id,name,cat,size,ts,te,dur,trange,mount_point,hostname,deg
7866,19208,open64,POSIX,31,7434222,7436811,2589,7,/usr/WS2,corona198,18
7867,19206,open64,POSIX,31,7434225,7436827,2602,7,/usr/WS2,corona198,18
7868,19206,open64,POSIX,30,7435034,7437162,2128,7,/usr/WS2,corona191,21
7869,19208,open64,POSIX,27,7435038,7437135,2097,7,/usr/WS2,corona191,21
7870,19206,open64,POSIX,31,7435044,7437530,2486,7,/usr/WS2,corona191,24
...,...,...,...,...,...,...,...,...,...,...,...
34144,127036,read,POSIX,14906,229999594,229999599,5,229,/usr/WS2,corona225,18
34145,127969,open64,POSIX,73,229999619,230005828,6209,229,/usr/WS2,corona191,21
34146,126246,open64,POSIX,73,229999665,230001682,2017,229,/usr/WS2,corona198,21
34147,127059,read,POSIX,37715,229999833,229999846,13,229,/usr/WS2,corona191,18


In [25]:
data.groupby(['trange']).count()

,id,name,cat,size,ts,te,dur,mount_point,hostname
trange,,,,,,,,,
0,123462,123462,123462,123462,123462,123462,123462,123462,123462
1,27345,27345,27345,27345,27345,27345,27345,27345,27345
2,72,72,72,72,72,72,72,72,72
3,144,144,144,144,144,144,144,144,144
5,55,55,55,55,55,55,55,55,55
...,...,...,...,...,...,...,...,...,...
284,11664,11664,11664,11664,11664,11664,11664,11664,11664
285,12636,12636,12636,12636,12636,12636,12636,12636,12636
286,7979,7979,7979,7979,7979,7979,7979,7979,7979


In [37]:
data.groupby(['trange','mount_point']).size().reset_index(name="Count").sort_values(by="Count",ascending = False).head(10)

,trange,mount_point,Count
101,201,/sys,197245
187,234,/usr/WS2,169241
184,233,/usr/WS2,108854
181,232,/usr/WS2,106201
106,202,/sys,96484
178,231,/usr/WS2,85982
54,23,/usr/WS2,75397
175,230,/usr/WS2,63699
172,229,/usr/WS2,62674
34,13,/usr/WS2,58982


In [38]:
metadata.groupby(['trange','mount_point']).size().reset_index(name="Count").sort_values(by="Count",ascending = False).head(10)

,trange,mount_point,Count
133,25,/usr/WS2,1921005
138,26,/usr/WS2,1857500
126,24,/usr/WS2,951111
166,33,/usr/WS2,876055
362,234,/usr/WS2,825034
175,34,/usr/WS2,539073
358,233,/usr/WS2,529408
15,2,/usr/WS2,517006
353,232,/usr/WS2,509263
11,2,/collab/usr/gapps,484608


In [ ]:
IFCalculator.get_interference()
IFCalculator.get_interference_metadata()

In [ ]:
IFCalculator.write_checkpoint(id="inter", cp_dir = cp_dir)
IFCalculator.write_checkpoint(id="inter_metadata", cp_dir = cp_dir)

In [12]:
IFCalculator.get_degree()
# IFCalculator.get_interference()